# BirdCLEF 2026 — Baseline: EfficientNet-B3 on Mel Spectrograms

Model 1 from the plan. End-to-end training run with the pipeline defined in `src/`.

To train from the terminal instead:
```bash
python src/train.py --model efficientnet_b3 --experiment exp001_baseline
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler
import matplotlib.pyplot as plt

from config import Config, NUM_CLASSES
from dataset import build_dataloaders
from augmentation import Augmentation, batch_mixup
from evaluate import evaluate
from models import build_model

# ── Configuration ──────────────────────────────────────────────────────────────
cfg = Config()
cfg.model_name      = 'efficientnet_b3'
cfg.experiment_name = 'exp001_baseline'
cfg.epochs          = 30
cfg.batch_size      = 32
cfg.learning_rate   = 1e-3
cfg.debug           = False  # set True for a quick smoke-test

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────────────
augment = Augmentation(cfg)
bird_loader, sc_loader, val_loader = build_dataloaders(cfg, augment=augment)
print(f'Bird batches/epoch: {len(bird_loader)}')
print(f'Soundscape batches/epoch: {len(sc_loader)}')
print(f'Val batches: {len(val_loader)}')

# Quick shape check
specs, labels = next(iter(bird_loader))
print(f'Spec shape: {specs.shape}   Label shape: {labels.shape}')

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
model = build_model(cfg.model_name, NUM_CLASSES, pretrained=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}  |  Trainable: {trainable:,}')

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs - cfg.warmup_epochs, eta_min=cfg.learning_rate * 0.01)
scaler    = GradScaler('cuda', enabled=(device.type == 'cuda'))

history = {'train_loss': [], 'cmap': []}
best_cmap = 0.0

for epoch in range(1, cfg.epochs + 1):
    model.train()
    total_loss = 0.0
    steps = 0
    bird_iter = iter(bird_loader)
    sc_iter   = iter(sc_loader)
    n_batches = max(len(bird_loader), len(sc_loader))

    for i in range(n_batches):
        use_bird = (i % round(1 / (1 - cfg.bird_soundscape_ratio + 1e-9))) != 0
        try:
            specs, labels = next(bird_iter if use_bird else sc_iter)
        except StopIteration:
            break

        specs  = specs.to(device)
        labels = labels.to(device)
        specs, labels = batch_mixup(specs, labels, alpha=cfg.mixup_alpha)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = model(specs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        steps += 1

    if epoch > cfg.warmup_epochs:
        scheduler.step()

    metrics = evaluate(model, val_loader, device)
    cmap = metrics['cmap']
    avg_loss = total_loss / max(steps, 1)
    history['train_loss'].append(avg_loss)
    history['cmap'].append(cmap)

    if cmap > best_cmap:
        best_cmap = cmap
        ckpt_dir = cfg.checkpoints_dir / cfg.experiment_name
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'best_cmap': best_cmap, 'cfg': cfg},
                   ckpt_dir / 'best.pt')

    print(f'Epoch {epoch:3d}  loss={avg_loss:.4f}  cmAP={cmap:.4f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training loss')
axes[0].legend()

axes[1].plot(history['cmap'], label='Val cmAP', color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('cmAP')
axes[1].set_title('Validation cmAP')
axes[1].legend()

plt.suptitle(f'EfficientNet-B3 Baseline — Best cmAP: {best_cmap:.4f}')
plt.tight_layout()
plt.show()

print(f'\nBest cmAP: {best_cmap:.4f}')
print(f'Checkpoint saved to: {cfg.checkpoints_dir / cfg.experiment_name / "best.pt"}')

In [ ]:
# ── Generate submission ────────────────────────────────────────────────────────
# Run predict.py from terminal:
# python src/predict.py --checkpoint checkpoints/exp001_baseline/best.pt --model efficientnet_b3
print('To generate a submission CSV run:')
print('  python src/predict.py --checkpoint checkpoints/exp001_baseline/best.pt')